# Explore the AI research source inventory

This notebook analyses the local `seed_urls.json` manifest only. It does not make network requests or download page content. Run it from the repository root or directly in JupyterLab.

Use this notebook to understand the source coverage before enabling the collector.

In [ ]:
from pathlib import Path
import json
import pandas as pd

candidates = [Path.cwd() / 'seed_urls.json', Path.cwd().parent / 'seed_urls.json']
manifest_path = next((p.resolve() for p in candidates if p.exists()), None)
if manifest_path is None:
    raise FileNotFoundError('Could not find seed_urls.json. Start Jupyter from the repository root or notebooks/ directory.')

rows = json.loads(manifest_path.read_text(encoding='utf-8'))
df = pd.DataFrame(rows)
print(f'Manifest: {manifest_path}')
print(f'Records: {len(df)}')
display(df.head())

## Inventory coverage
Count manifest entries by person/source and resource type. These are URL records, not a count of downloaded documents.

In [ ]:
person_col = 'Person / source'
type_col = 'Resource type'
priority_col = 'Priority'
display(df.groupby(person_col, dropna=False).size().rename('url_records').reset_index())
display(df.groupby([person_col, type_col], dropna=False).size().rename('url_records').reset_index().sort_values(['Person / source', 'url_records'], ascending=[True, False]))
display(df.groupby([person_col, priority_col], dropna=False).size().rename('url_records').reset_index())

## High-priority records for manual review
Use this filtered view to check the first pages and document whether policy, content type, and reuse rights are understood before collection.

In [ ]:
high_priority = df[df[priority_col].astype(str).str.upper().isin(['P1', 'P2'])].copy()
cols = [c for c in [ 'Manifest ID', person_col, 'URL', 'Page / resource name', type_col, priority_col, 'Verification status', 'Robots / sitemap status', 'Rights status', 'Collection decision'] if c in high_priority.columns]
display(high_priority[cols].sort_values([priority_col, person_col, 'Manifest ID']))

## Readiness questions
Before enabling a network run, verify the current `robots.txt` rules from your own network, keep the user-agent descriptive, and review content reuse terms. A `P1` priority is a project prioritization, not permission to fetch or republish content.